In [0]:
total = spark.range(3).count()

print(f"Total de registros: {total}")

In [0]:
volume_path = "/Volumes/workspace/amazon_reviews/raw_files"

pastas = ["", "reviews", "products"]

for pasta in pastas:
    caminho = f"{volume_path}/{pasta}"
    itens = dbutils.fs.ls(caminho)

    print(f"\nDiretório: {caminho}")
    print([item.name for item in itens])

In [0]:
caminho = "/Volumes/workspace/amazon_reviews/raw_files/_teste_escrita_1_11.txt"

conteudo = "Teste de escrita no Unity Catalog"

# Criar arquivo
gravado = dbutils.fs.put(caminho, conteudo, overwrite=False)
print("Gravação:", gravado)

# Verificar conteúdo
lido = dbutils.fs.head(caminho, 100)
print("Leitura correta:", lido == conteudo)

# Excluir arquivo temporário
if gravado and lido == conteudo:
    removido = dbutils.fs.rm(caminho, recurse=False)
    print("Remoção:", removido)

In [0]:
# Caminho principal do Volume
base_path = "/Volumes/workspace/amazon_reviews/raw_files"

# Caminhos dos arquivos
reviews_path = f"{base_path}/reviews/reviews_all_beauty.parquet"
products_path = f"{base_path}/products/products_all_beauty.parquet"

# Leitura dos arquivos como DataFrames
df_reviews = spark.read.parquet(reviews_path)
df_products = spark.read.parquet(products_path)

# Inspeção dos schemas
print("=== SCHEMA REVIEWS ===")
df_reviews.printSchema()

print("\n=== SCHEMA PRODUCTS ===")
df_products.printSchema()

# Verificação da chave de relacionamento
print("\n=== CHAVE DE RELACIONAMENTO ===")
print("Reviews:", "parent_asin" in df_reviews.columns)
print("Products:", "parent_asin" in df_products.columns)

In [0]:
# Amostra de avaliações
print("=== AMOSTRA REVIEWS ===")

df_reviews.select(
    "parent_asin",
    "asin",
    "rating",
    "timestamp",
    "helpful_vote",
    "verified_purchase"
).limit(5).show(truncate=40)


# Amostra de produtos
print("=== AMOSTRA PRODUCTS ===")

df_products.select(
    "parent_asin",
    "title",
    "average_rating",
    "rating_number",
    "price",
    "store"
).limit(5).show(truncate=40)

In [0]:
from pyspark.sql import functions as F

# Indicadores de Reviews
quality_reviews = df_reviews.agg(
    F.count("*").alias("total_reviews"),

    F.count(
        F.when(
            F.col("parent_asin").isNull() |
            (F.trim(F.col("parent_asin")) == ""),
            1
        )
    ).alias("reviews_sem_parent_asin")
)

# Indicadores de Products
quality_products = df_products.agg(
    F.count("*").alias("total_products"),

    F.count(
        F.when(
            F.col("parent_asin").isNull() |
            (F.trim(F.col("parent_asin")) == ""),
            1
        )
    ).alias("products_sem_parent_asin"),

    F.count(
        F.when(F.col("price").isNull(), 1)
    ).alias("price_null"),

    F.count(
        F.when(
            F.lower(F.trim(F.col("price"))).isin(
                "none", "null", ""
            ),
            1
        )
    ).alias("price_texto_ausente")
)

print("=== QUALIDADE REVIEWS ===")
quality_reviews.show(truncate=False)

print("=== QUALIDADE PRODUCTS ===")
quality_products.show(truncate=False)

In [0]:
from pyspark.sql import functions as F

# Contar ocorrências de cada produto
products_por_chave = (
    df_products
    .groupBy("parent_asin")
    .count()
)

# Resumir a cardinalidade
cardinalidade_products = products_por_chave.agg(
    F.count("*").alias("chaves_distintas"),

    F.sum(
        F.when(F.col("count") > 1, 1).otherwise(0)
    ).alias("chaves_duplicadas"),

    F.sum(
        F.when(
            F.col("count") > 1,
            F.col("count") - 1
        ).otherwise(0)
    ).alias("registros_excedentes"),

    F.max("count").alias("max_registros_por_chave")
)

print("=== CARDINALIDADE PRODUCTS ===")
cardinalidade_products.show(truncate=False)

In [0]:
from pyspark.sql import functions as F

# Selecionar somente a chave dos produtos
chaves_produtos = df_products.select(
    F.col("parent_asin").alias("produto_parent_asin")
)

# Relacionar avaliações aos produtos
reviews_com_produtos = (
    df_reviews.select("parent_asin")
    .join(
        chaves_produtos,
        F.col("parent_asin") == F.col("produto_parent_asin"),
        "left"
    )
)

# Calcular indicadores de correspondência
resultado_join = reviews_com_produtos.agg(
    F.count("*").alias("total_reviews_apos_join"),

    F.count("produto_parent_asin").alias(
        "reviews_com_produto"
    ),

    F.sum(
        F.when(
            F.col("produto_parent_asin").isNull(), 1
        ).otherwise(0)
    ).alias("reviews_sem_produto")
)

print("=== CORRESPONDÊNCIA REVIEWS X PRODUCTS ===")
resultado_join.show(truncate=False)

In [0]:
from pyspark.sql import functions as F

# Agrupar avaliações por combinação de identificação
reviews_por_chave = (
    df_reviews
    .groupBy("user_id", "asin", "timestamp")
    .count()
)

# Calcular indicadores de possíveis duplicações
duplicacoes_reviews = reviews_por_chave.agg(
    F.count("*").alias("combinacoes_distintas"),

    F.sum(
        F.when(F.col("count") > 1, 1).otherwise(0)
    ).alias("combinacoes_repetidas"),

    F.sum(
        F.when(
            F.col("count") > 1,
            F.col("count") - 1
        ).otherwise(0)
    ).alias("registros_excedentes"),

    F.max("count").alias("max_por_combinacao")
)

print("=== POSSIVEIS DUPLICACOES REVIEWS ===")
duplicacoes_reviews.show(truncate=False)

In [0]:
from pyspark.sql import functions as F

# Encontrar o grupo com mais repetições
grupo = (
    reviews_por_chave
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
    .limit(1)
    .collect()[0]
)

print("=== GRUPO MAIS REPETIDO ===")
print("Quantidade:", grupo["count"])

# Recuperar as avaliações pertencentes ao grupo
amostra_duplicacoes = df_reviews.filter(
    F.col("user_id").eqNullSafe(grupo["user_id"]) &
    F.col("asin").eqNullSafe(grupo["asin"]) &
    F.col("timestamp").eqNullSafe(grupo["timestamp"])
)

# Exibir os campos relevantes
amostra_duplicacoes.select(
    "parent_asin",
    "rating",
    "title",
    "text",
    "helpful_vote",
    "verified_purchase"
).show(20, truncate=60)

In [0]:
# Quantidade de registros do grupo
total_registros = grupo["count"]

# Quantidade de registros completamente distintos
registros_distintos = amostra_duplicacoes.distinct().count()

# Quantidade de registros excedentes
registros_excedentes = total_registros - registros_distintos

print("=== VALIDACAO DE DUPLICACOES EXATAS ===")
print("Total de registros:", total_registros)
print("Registros distintos:", registros_distintos)
print("Registros excedentes:", registros_excedentes)

In [0]:
from pyspark.sql import functions as F

# 1. Identificar combinações repetidas
chaves_repetidas = (
    reviews_por_chave
    .filter(F.col("count") > 1)
    .select("user_id", "asin", "timestamp")
)

# 2. Recuperar somente as avaliações suspeitas
r = df_reviews.alias("r")
k = chaves_repetidas.alias("k")

condicao = (
    F.col("r.user_id").eqNullSafe(F.col("k.user_id")) &
    F.col("r.asin").eqNullSafe(F.col("k.asin")) &
    F.col("r.timestamp").eqNullSafe(F.col("k.timestamp"))
)

reviews_suspeitas = r.join(
    k,
    condicao,
    "left_semi"
)

# 3. Contar os registros suspeitos
total_suspeitos = reviews_suspeitas.count()

# 4. Contar os registros completamente distintos
total_distintos = reviews_suspeitas.distinct().count()

# 5. Calcular duplicações exatas
duplicacoes_exatas = total_suspeitos - total_distintos

print("=== DUPLICACOES EXATAS REVIEWS ===")
print("Registros suspeitos:", total_suspeitos)
print("Registros distintos:", total_distintos)
print("Duplicações exatas:", duplicacoes_exatas)

In [0]:
from pyspark.sql import functions as F

# 1. Manter registros completamente distintos
reviews_distintas = reviews_suspeitas.distinct()

# 2. Identificar combinações com versões diferentes
grupos_divergentes = (
    reviews_distintas
    .groupBy("user_id", "asin", "timestamp")
    .count()
    .filter(F.col("count") > 1)
    .select("user_id", "asin", "timestamp")
)

# 3. Recuperar as versões divergentes
r = reviews_distintas.alias("r")
g = grupos_divergentes.alias("g")

condicao = (
    F.col("r.user_id").eqNullSafe(F.col("g.user_id")) &
    F.col("r.asin").eqNullSafe(F.col("g.asin")) &
    F.col("r.timestamp").eqNullSafe(F.col("g.timestamp"))
)

versoes = r.join(g, condicao, "left_semi").limit(3).collect()

# 4. Comparar os registros
print("=== INVESTIGACAO DE DIVERGENCIAS ===")
print("Versões distintas encontradas:", len(versoes))

if len(versoes) == 2:
    for campo in df_reviews.columns:
        valor_1 = versoes[0][campo]
        valor_2 = versoes[1][campo]

        if valor_1 != valor_2:
            print(f"\nCampo divergente: {campo}")
            print("Versão 1:", str(valor_1)[:150])
            print("Versão 2:", str(valor_2)[:150])
else:
    print("Quantidade inesperada. Investigar antes de continuar.")

In [0]:
from pyspark.sql import functions as F

qualidade_reviews = df_reviews.agg(

    F.count("*").alias("total_reviews"),

    F.sum(
        F.when(F.col("rating").isNull(), 1).otherwise(0)
    ).alias("rating_nulo"),

    F.sum(
        F.when(
            (F.col("rating") < 1) |
            (F.col("rating") > 5),
            1
        ).otherwise(0)
    ).alias("rating_fora_intervalo"),

    F.min("rating").alias("rating_min"),
    F.max("rating").alias("rating_max"),

    F.sum(
        F.when(
            F.col("helpful_vote").isNull() |
            (F.col("helpful_vote") < 0),
            1
        ).otherwise(0)
    ).alias("helpful_vote_invalido"),

    F.sum(
        F.when(
            F.col("timestamp").isNull() |
            (F.col("timestamp") <= 0),
            1
        ).otherwise(0)
    ).alias("timestamp_invalido"),

    F.min("timestamp").alias("timestamp_min"),
    F.max("timestamp").alias("timestamp_max")

)

print("=== QUALIDADE NUMERICA REVIEWS ===")
qualidade_reviews.show(truncate=False, vertical=True)

In [0]:
from pyspark.sql import functions as F

# Padronizar as datas em UTC
spark.conf.set("spark.sql.session.timeZone", "UTC")

# Converter timestamp e extrair o ano
reviews_por_ano = (
    df_reviews
    .select(
        F.year(
            F.timestamp_millis("timestamp")
        ).alias("ano")
    )
    .groupBy("ano")
    .agg(
        F.count("*").alias("total_avaliacoes")
    )
    .orderBy("ano")
)

print("=== DISTRIBUICAO ANUAL REVIEWS ===")
reviews_por_ano.show(30, truncate=False)

In [0]:
from pyspark.sql import functions as F

distribuicao_notas = (
    df_reviews
    .groupBy("rating")
    .agg(
        F.count("*").alias("total_avaliacoes"),

        F.sum(
            F.when(
                F.col("verified_purchase") == True, 1
            ).otherwise(0)
        ).alias("compras_verificadas"),

        F.sum(
            F.when(
                F.col("verified_purchase") == False, 1
            ).otherwise(0)
        ).alias("compras_nao_verificadas"),

        F.sum(
            F.when(
                F.col("verified_purchase").isNull(), 1
            ).otherwise(0)
        ).alias("verificacao_nula")
    )
    .orderBy("rating")
)

print("=== DISTRIBUICAO DAS NOTAS ===")
distribuicao_notas.show(20, truncate=False)

In [0]:
from pyspark.sql import functions as F

qualidade_products = df_products.agg(

    F.count("*").alias("total_products"),

    # Validar média das notas
    F.sum(
        F.when(
            F.col("average_rating").isNull(), 1
        ).otherwise(0)
    ).alias("average_rating_nulo"),

    F.sum(
        F.when(
            (F.col("average_rating") < 0) |
            (F.col("average_rating") > 5),
            1
        ).otherwise(0)
    ).alias("average_rating_fora_0_5"),

    F.sum(
        F.when(
            F.col("average_rating") == 0, 1
        ).otherwise(0)
    ).alias("average_rating_zero"),

    F.min("average_rating").alias("average_rating_min"),
    F.max("average_rating").alias("average_rating_max"),

    # Validar quantidade de avaliações
    F.sum(
        F.when(
            F.col("rating_number").isNull(), 1
        ).otherwise(0)
    ).alias("rating_number_nulo"),

    F.sum(
        F.when(
            F.col("rating_number") < 0, 1
        ).otherwise(0)
    ).alias("rating_number_negativo"),

    F.sum(
        F.when(
            F.col("rating_number") == 0, 1
        ).otherwise(0)
    ).alias("rating_number_zero"),

    F.max("rating_number").alias("rating_number_max")
)

print("=== QUALIDADE NUMERICA PRODUCTS ===")
qualidade_products.show(truncate=False, vertical=True)

In [0]:
from pyspark.sql import functions as F

# 1. Contar avaliações por produto
reviews_por_produto = (
    df_reviews
    .groupBy("parent_asin")
    .agg(
        F.count("*").alias("reviews_no_parquet")
    )
)

# 2. Relacionar as contagens aos metadados
comparacao_produtos = (
    df_products
    .select("parent_asin", "rating_number")
    .join(
        reviews_por_produto,
        on="parent_asin",
        how="left"
    )
    .withColumn(
        "reviews_no_parquet",
        F.coalesce(F.col("reviews_no_parquet"), F.lit(0))
    )
)

# 3. Calcular os indicadores
resultado = comparacao_produtos.agg(

    F.count("*").alias("total_produtos"),

    F.sum(
        F.when(
            F.col("reviews_no_parquet") > 0, 1
        ).otherwise(0)
    ).alias("produtos_com_reviews"),

    F.sum(
        F.when(
            F.col("reviews_no_parquet") == 0, 1
        ).otherwise(0)
    ).alias("produtos_sem_reviews"),

    F.sum(
        F.when(
            F.col("rating_number") == F.col("reviews_no_parquet"), 1
        ).otherwise(0)
    ).alias("quantidades_iguais"),

    F.sum(
        F.when(
            F.col("rating_number") != F.col("reviews_no_parquet"), 1
        ).otherwise(0)
    ).alias("quantidades_diferentes"),

    F.sum("reviews_no_parquet").alias("reviews_total_conciliado")
)

print("=== COMPARACAO PRODUCTS X REVIEWS ===")
resultado.show(truncate=False, vertical=True)

In [0]:
from pyspark.sql import functions as F

# Calcular a diferença entre as duas contagens
comparacao_diferencas = comparacao_produtos.withColumn(
    "diferenca",
    F.col("rating_number") - F.col("reviews_no_parquet")
)

# Resumir as diferenças
resultado_diferencas = comparacao_diferencas.agg(

    F.sum("rating_number").alias(
        "total_rating_number_origem"
    ),

    F.sum("reviews_no_parquet").alias(
        "total_reviews_parquet"
    ),

    F.sum(
        F.when(F.col("diferenca") > 0, 1).otherwise(0)
    ).alias("origem_maior"),

    F.sum(
        F.when(F.col("diferenca") < 0, 1).otherwise(0)
    ).alias("parquet_maior"),

    F.sum(
        F.when(F.col("diferenca") == 0, 1).otherwise(0)
    ).alias("quantidades_iguais"),

    F.sum("diferenca").alias("saldo_diferenca"),

    F.max(
        F.abs(F.col("diferenca"))
    ).alias("maior_diferenca_absoluta")
)

print("=== ANALISE DAS DIVERGENCIAS ===")
resultado_diferencas.show(
    truncate=False,
    vertical=True
)

In [0]:
from pyspark.sql import functions as F

# Colunas necessárias para a investigação
colunas = [
    "parent_asin",
    "rating_number",
    "reviews_no_parquet",
    "diferenca"
]

# Produtos com maiores diferenças positivas
print("=== MAIORES DIFERENCAS POSITIVAS ===")

(
    comparacao_diferencas
    .filter(F.col("diferenca") > 0)
    .select(*colunas)
    .orderBy(F.desc("diferenca"))
    .limit(5)
    .show(truncate=False)
)

# Produtos com maiores diferenças negativas
print("=== MAIORES DIFERENCAS NEGATIVAS ===")

(
    comparacao_diferencas
    .filter(F.col("diferenca") < 0)
    .select(*colunas)
    .orderBy(F.asc("diferenca"))
    .limit(5)
    .show(truncate=False)
)

In [0]:
from pyspark.sql import functions as F

# 1. Produtos que apresentaram divergências negativas
produtos_alvo = [
    "B0BTLTVR1X",
    "B07R1K1SBD",
    "B09FR54XLC",
    "B08BBQ29N5",
    "B086H4C8QB"
]

# 2. Selecionar somente as avaliações desses produtos
reviews_alvo = df_reviews.filter(
    F.col("parent_asin").isin(produtos_alvo)
)

# 3. Contar as avaliações brutas
contagens_brutas = (
    reviews_alvo
    .groupBy("parent_asin")
    .agg(
        F.count("*").alias("reviews_brutas")
    )
)

# 4. Contar somente os registros distintos
contagens_distintas = (
    reviews_alvo
    .distinct()
    .groupBy("parent_asin")
    .agg(
        F.count("*").alias("reviews_distintas")
    )
)

# 5. Comparar os resultados com Products
resultado_dedup = (
    df_products
    .filter(F.col("parent_asin").isin(produtos_alvo))
    .select("parent_asin", "rating_number")
    .join(contagens_brutas, "parent_asin", "left")
    .join(contagens_distintas, "parent_asin", "left")
    .withColumn(
        "copias_exatas",
        F.col("reviews_brutas") - F.col("reviews_distintas")
    )
    .withColumn(
        "diferenca_apos_dedup",
        F.col("rating_number") - F.col("reviews_distintas")
    )
    .orderBy("parent_asin")
)

print("=== IMPACTO DAS DUPLICACOES ===")
resultado_dedup.show(truncate=False)

In [0]:
from pyspark.sql import functions as F

# Ler a tabela Bronze
df_bronze = spark.table(
    "workspace.amazon_reviews.bronze_reviews_raw"
)

# Calcular indicadores de validação
validacao_bronze = df_bronze.agg(
    F.count("*").alias("total_registros"),

    F.count("_source_file").alias(
        "registros_com_origem"
    ),

    F.count("_source_file_modified_at").alias(
        "registros_com_data_arquivo"
    ),

    F.count("_ingested_at").alias(
        "registros_com_ingestao"
    )
)

print("=== VALIDACAO BRONZE REVIEWS ===")
validacao_bronze.show(truncate=False)

In [0]:
from pyspark.sql import functions as F

bronze = spark.table(
    "workspace.amazon_reviews.bronze_reviews_raw"
)

estado_bronze = bronze.agg(
    F.count("*").alias("total_registros"),
    F.min("_ingested_at").alias("primeira_ingestao"),
    F.max("_ingested_at").alias("ultima_ingestao")
)

print("=== ESTADO BRONZE REVIEWS ===")
estado_bronze.show(truncate=False)

In [0]:
from pyspark.sql import functions as F

# Ler a tabela Bronze Products
df_bronze_products = spark.table(
    "workspace.amazon_reviews.bronze_products_raw"
)

# Calcular indicadores de ingestão
validacao_products = df_bronze_products.agg(

    F.count("*").alias("total_registros"),

    F.count("_source_file").alias(
        "registros_com_origem"
    ),

    F.count("_source_file_modified_at").alias(
        "registros_com_data_arquivo"
    ),

    F.count("_ingested_at").alias(
        "registros_com_ingestao"
    )
)

print("=== VALIDACAO BRONZE PRODUCTS ===")
validacao_products.show(truncate=False)

In [0]:
total_reviews = spark.table(
    "workspace.amazon_reviews.bronze_reviews_raw"
).count()

print("=== VERIFICACAO BRONZE REVIEWS ===")
print("Total de avaliações:", total_reviews)

In [0]:
from pyspark.sql import functions as F

# Ler as duas tabelas Bronze
df_products_bronze = spark.table(
    "workspace.amazon_reviews.bronze_products_raw"
)

df_reviews_bronze = spark.table(
    "workspace.amazon_reviews.bronze_reviews_raw"
)

# Verificar Products
print("=== ESTADO BRONZE PRODUCTS ===")

df_products_bronze.agg(
    F.count("*").alias("total_products"),
    F.min("_ingested_at").alias("primeira_ingestao"),
    F.max("_ingested_at").alias("ultima_ingestao")
).show(truncate=False)

# Verificar Reviews
print("=== ESTADO BRONZE REVIEWS ===")

df_reviews_bronze.agg(
    F.count("*").alias("total_reviews")
).show(truncate=False)

In [0]:
from pyspark.sql import functions as F

tabelas = [
    "bronze_reviews_raw",
    "bronze_products_raw"
]

for tabela in tabelas:

    df = spark.table(
        f"workspace.amazon_reviews.{tabela}"
    )

    print(f"=== VALIDACAO {tabela.upper()} ===")

    if "_rescued_data" in df.columns:

        df.agg(
            F.count("*").alias("total_registros"),

            F.count("_rescued_data").alias(
                "registros_resgatados"
            )

        ).show(truncate=False)

    else:
        print("Coluna _rescued_data não encontrada")

In [0]:
from pyspark.sql import functions as F

# Ler as tabelas Bronze e Silver
df_bronze = spark.table(
    "workspace.amazon_reviews.bronze_reviews_raw"
)

df_silver = spark.table(
    "workspace.amazon_reviews.silver_reviews_clean"
)

# Contagem original
total_bronze = df_bronze.count()

# Indicadores da Silver
resultado = df_silver.agg(
    F.count("*").alias("total_silver"),

    F.sum(
        F.when(
            F.col("review_timestamp_utc").isNull(), 1
        ).otherwise(0)
    ).alias("timestamps_nulos"),

    F.sum(
        F.when(
            F.col("rating").isNull()
            | ~F.col("rating").between(1, 5),
            1
        ).otherwise(0)
    ).alias("ratings_invalidos"),

    F.sum(
        F.when(
            F.col("helpful_vote").isNull()
            | (F.col("helpful_vote") < 0),
            1
        ).otherwise(0)
    ).alias("votos_invalidos")
).collect()[0]

print("=== VALIDACAO SILVER REVIEWS ===")

print("Total Bronze:", total_bronze)
print("Total Silver:", resultado["total_silver"])

print(
    "Duplicatas removidas:",
    total_bronze - resultado["total_silver"]
)

print(
    "Timestamps nulos:",
    resultado["timestamps_nulos"]
)

print(
    "Ratings invalidos:",
    resultado["ratings_invalidos"]
)

print(
    "Helpful votes invalidos:",
    resultado["votos_invalidos"]
)

In [0]:
from pyspark.sql import functions as F

df_silver = spark.table(
    "workspace.amazon_reviews.silver_reviews_clean"
)

# Converter o timestamp novamente para milissegundos
# e comparar com o valor original.
timestamps_divergentes = df_silver.filter(
    F.unix_millis("review_timestamp_utc")
    != F.col("timestamp")
).count()

print("=== INTEGRIDADE DOS TIMESTAMPS ===")
print("Timestamps divergentes:", timestamps_divergentes)

In [0]:
# Agrupar pela chave operacional
chaves_repetidas = (
    df_silver
    .groupBy("user_id", "asin", "timestamp")
    .count()
    .filter(F.col("count") > 1)
)

# Resumir as ocorrências
resultado = chaves_repetidas.agg(
    F.count("*").alias("chaves_repetidas"),
    F.coalesce(
        F.sum("count"), F.lit(0)
    ).alias("registros_preservados")
)

print("=== VALIDACAO DE VERSOES DISTINTAS ===")
resultado.show(truncate=False)

In [0]:
from pyspark.sql import functions as F

df_products = spark.table(
    "workspace.amazon_reviews.bronze_products_raw"
)

# Remover espaços nas extremidades
df_precos = df_products.withColumn(
    "preco_texto",
    F.trim(F.col("price"))
)

# Classificar os formatos encontrados
df_precos = df_precos.withColumn(
    "categoria_preco",

    F.when(
        F.col("price").isNull()
        | F.lower(F.col("preco_texto")).isin(
            "none", "null", ""
        ),
        "ausente"
    )

    .when(
        F.col("preco_texto").rlike(
            r"^\d+(\.\d+)?$"
        ),
        "numerico_simples"
    )

    .when(
        F.col("preco_texto").rlike(
            r"^\$\s*\d+(\.\d+)?$"
        ),
        "com_dolar"
    )

    .when(
        F.col("preco_texto").rlike(
            r"^\$?\s*\d{1,3}(,\d{3})+(\.\d+)?$"
        ),
        "separador_milhar"
    )

    .otherwise("outros")
)

print("=== FORMATOS DE PRECO ===")

df_precos.groupBy("categoria_preco").agg(
    F.count("*").alias("quantidade")
).orderBy(
    F.desc("quantidade")
).show(truncate=False)

print("=== EXEMPLOS DE FORMATOS NAO RECONHECIDOS ===")

df_precos.filter(
    F.col("categoria_preco") == "outros"
).groupBy("price").count().orderBy(
    F.desc("count")
).show(20, truncate=False)

In [0]:
from pyspark.sql import functions as F

# Ler Bronze e Silver Products
df_bronze = spark.table(
    "workspace.amazon_reviews.bronze_products_raw"
)

df_silver = spark.table(
    "workspace.amazon_reviews.silver_products_clean"
)

print("=== VALIDACAO SILVER PRODUCTS ===")

print("Total Bronze:", df_bronze.count())

df_silver.agg(
    F.count("*").alias("total_silver"),

    F.countDistinct("parent_asin").alias(
        "produtos_distintos"
    ),

    F.sum(
        F.when(
            F.col("price_status") == "valido", 1
        ).otherwise(0)
    ).alias("precos_validos"),

    F.sum(
        F.when(
            F.col("price_status") == "ausente", 1
        ).otherwise(0)
    ).alias("precos_ausentes"),

    F.sum(
        F.when(
            F.col("price_status") == "invalido", 1
        ).otherwise(0)
    ).alias("precos_invalidos"),

    F.count("price").alias("precos_numericos"),

    F.sum(
        F.when(
            F.col("price_status").isNull(), 1
        ).otherwise(0)
    ).alias("status_nulo")

).show(truncate=False)

In [0]:
print("=== SCHEMA DOS PRECOS ===")

df_silver.select(
    "price_raw",
    "price",
    "price_status"
).printSchema()

In [0]:
print("=== VERIFICACAO REVIEWS ===")

print(
    "Bronze Reviews:",
    spark.table(
        "workspace.amazon_reviews.bronze_reviews_raw"
    ).count()
)

print(
    "Silver Reviews:",
    spark.table(
        "workspace.amazon_reviews.silver_reviews_clean"
    ).count()
)

In [0]:
from pyspark.sql import functions as F

df = spark.table(
    "workspace.amazon_reviews.silver_products_clean"
)

# Identificar precos originalmente ausentes
preco_ausente = (
    F.col("price_raw").isNull()
    | F.lower(F.trim("price_raw")).isin(
        "none", "null", ""
    )
)

# Recalcular o preco para comparar os valores
preco_recalculado = F.expr(
    "try_cast(trim(price_raw) AS DECIMAL(12,2))"
)

# Conferir as regras de qualidade
validacao = df.agg(

    F.count("*").alias("total_produtos"),

    F.countDistinct("parent_asin").alias(
        "produtos_distintos"
    ),

    F.sum(F.when(
        F.col("parent_asin").isNull()
        | (F.trim("parent_asin") == ""), 1
    ).otherwise(0)).alias("ids_invalidos"),

    F.sum(F.when(
        F.col("average_rating").isNull()
        | ~F.col("average_rating").between(0, 5), 1
    ).otherwise(0)).alias("ratings_invalidos"),

    F.sum(F.when(
        F.col("rating_number").isNull()
        | (F.col("rating_number") < 0), 1
    ).otherwise(0)).alias("contagens_invalidas"),

    F.sum(F.when(
        (preco_ausente & (F.col("price_status") != "ausente"))
        | (~preco_ausente & (F.col("price_status") != "valido"))
        | F.col("price_status").isNull(), 1
    ).otherwise(0)).alias("status_incoerentes"),

    F.sum(F.when(
        ~F.col("price").eqNullSafe(preco_recalculado), 1
    ).otherwise(0)).alias("precos_divergentes"),

    F.sum(F.when(
        F.col("price") < 0, 1
    ).otherwise(0)).alias("precos_negativos"),

    F.min("price").alias("menor_preco"),
    F.max("price").alias("maior_preco")
)

print("=== QUALIDADE SILVER PRODUCTS ===")
validacao.show(truncate=False)

In [0]:

from pyspark.sql import functions as F

df_gold = spark.table(
    "workspace.amazon_reviews.gold_product_metrics"
)

df_silver = spark.table(
    "workspace.amazon_reviews.silver_reviews_clean"
)

print("=== VALIDACAO GOLD PRODUCT METRICS ===")

df_gold.agg(
    F.count("*").alias("total_produtos"),

    F.countDistinct("parent_asin").alias(
        "produtos_distintos"
    ),

    F.sum("reviews_count").alias(
        "total_reviews_gold"
    ),

    F.sum("verified_review_count").alias(
        "total_verificadas_gold"
    ),

    F.sum("total_helpful_votes").alias(
        "total_votos_gold"
    ),

    F.sum(
        F.when(
            F.col("parent_asin").isNull()
            | (F.col("reviews_count") <= 0)
            | F.col("average_review_rating").isNull()
            | ~F.col("average_review_rating").between(1, 5)
            | (F.col("verified_review_count") < 0)
            | (
                F.col("verified_review_count")
                > F.col("reviews_count")
            )
            | F.col("verified_review_pct").isNull()
            | ~F.col("verified_review_pct").between(0, 100)
            | F.col("total_helpful_votes").isNull()
            | (F.col("total_helpful_votes") < 0),
            1
        ).otherwise(0)
    ).alias("metricas_invalidas")

).show(truncate=False)


print("=== CONFERENCIA SILVER REVIEWS ===")

df_silver.agg(
    F.count("*").alias("total_reviews_silver"),

    F.sum(
        F.when(
            F.col("verified_purchase") == True, 1
        ).otherwise(0)
    ).alias("total_verificadas_silver"),

    F.sum("helpful_vote").alias(
        "total_votos_silver"
    )
).show(truncate=False)


In [0]:

from pyspark.sql import functions as F

# Ler as tabelas Gold e Silver
df_gold = spark.table(
    "workspace.amazon_reviews.gold_monthly_review_metrics"
)

df_silver = spark.table(
    "workspace.amazon_reviews.silver_reviews_clean"
)

print("=== VALIDACAO GOLD MONTHLY METRICS ===")

df_gold.agg(
    F.count("*").alias("meses_com_avaliacoes"),

    F.countDistinct("review_month").alias(
        "meses_distintos"
    ),

    F.min("review_month").alias("primeiro_mes"),

    F.max("review_month").alias("ultimo_mes"),

    F.sum("reviews_count").alias(
        "total_reviews_gold"
    ),

    F.sum("verified_review_count").alias(
        "total_verificadas_gold"
    ),

    F.sum(
        F.when(
            F.col("review_month").isNull()
            | (F.dayofmonth("review_month") != 1)
            | (F.col("reviews_count") <= 0)
            | (F.col("reviewed_products_count") <= 0)
            | (
                F.col("reviewed_products_count")
                > F.col("reviews_count")
            )
            | F.col("average_review_rating").isNull()
            | ~F.col("average_review_rating").between(1, 5)
            | (F.col("verified_review_count") < 0)
            | (
                F.col("verified_review_count")
                > F.col("reviews_count")
            )
            | F.col("verified_review_pct").isNull()
            | ~F.col("verified_review_pct").between(0, 100),
            1
        ).otherwise(0)
    ).alias("metricas_invalidas")

).show(truncate=False)


print("=== CONFERENCIA SILVER REVIEWS ===")

df_silver.agg(
    F.count("*").alias("total_reviews_silver"),

    F.sum(
        F.when(
            F.col("verified_purchase") == True, 1
        ).otherwise(0)
    ).alias("total_verificadas_silver")
).show(truncate=False)


In [0]:
print("=== ULTIMOS MESES ===")

df_gold.orderBy(
    F.desc("review_month")
).show(5, truncate=False)

In [0]:

from pyspark.sql import functions as F

# Ler a Gold preparada para o Genie
df_genie = spark.table(
    "workspace.amazon_reviews.gold_genie_product_analytics"
)

print("=== VALIDACAO GOLD GENIE ===")

df_genie.agg(
    F.count("*").alias("total_produtos"),

    F.countDistinct("parent_asin").alias(
        "produtos_distintos"
    ),

    F.sum(
        F.when(F.col("reviews_count") > 0, 1)
         .otherwise(0)
    ).alias("produtos_com_reviews"),

    F.sum(
        F.when(F.col("reviews_count") == 0, 1)
         .otherwise(0)
    ).alias("produtos_sem_reviews"),

    F.sum("reviews_count").alias(
        "total_reviews"
    ),

    F.sum("verified_review_count").alias(
        "total_verificadas"
    ),

    F.sum("total_helpful_votes").alias(
        "total_votos"
    ),

    F.count("price").alias(
        "precos_numericos"
    ),

    F.sum(
        F.when(
            F.col("reviews_count").isNull()
            | (F.col("reviews_count") < 0)
            | (F.col("verified_review_count") < 0)
            | (
                F.col("verified_review_count")
                > F.col("reviews_count")
            )
            | (F.col("total_helpful_votes") < 0)
            | (
                (F.col("reviews_count") == 0)
                & (
                    F.col("average_review_rating").isNotNull()
                    | F.col("verified_review_pct").isNotNull()
                    | (F.col("verified_review_count") != 0)
                    | (F.col("total_helpful_votes") != 0)
                )
            )
            | (
                (F.col("reviews_count") > 0)
                & (
                    F.col("average_review_rating").isNull()
                    | ~F.col("average_review_rating").between(1, 5)
                    | F.col("verified_review_pct").isNull()
                    | ~F.col("verified_review_pct").between(0, 100)
                )
            ),
            1
        ).otherwise(0)
    ).alias("metricas_inconsistentes")

).show(truncate=False)


In [0]:
print("=== PRODUTOS SEM AVALIACOES ===")

df_genie.filter(
    F.col("reviews_count") == 0
).select(
    "parent_asin",
    "product_title",
    "source_rating_number",
    "reviews_count",
    "average_review_rating",
    "verified_review_pct"
).show(5, truncate=False)